# Qué hace `attrgetter("tool_calls")`

En `herramientas_personalizadas_llm.py` la cadena es esta:

```python
chain = llm_with_tools | attrgetter("tool_calls") | herramienta_personalizada.map()
```

`attrgetter("tool_calls")` es solo esta función, ya escrita por Python:

```python
def sacar_tool_calls(mensaje):
    return mensaje.tool_calls
```

No llama al modelo ni ejecuta la herramienta. Cuando el mensaje llega, lee el atributo `tool_calls`.

Abajo el mismo recorrido tres veces. Las tres devuelven la misma lista.

## El mensaje que imita al modelo

`llm_with_tools.invoke(...)` no devuelve el texto de la consulta. Devuelve un mensaje, y dentro va la lista `tool_calls`.

Estas funciones imitan ese mensaje, el modelo y la herramienta, sin llamar a la API.

In [ ]:
from operator import attrgetter


class Mensaje:
    """Imita el AIMessage que devuelve llm_with_tools.invoke(...)."""

    def __init__(self, tool_calls):
        self.tool_calls = tool_calls


def modelo(pregunta):
    """Imita llm_with_tools: devuelve un mensaje con tool_calls."""
    return Mensaje(
        tool_calls=[
            {
                "name": "user_db_tool",
                "args": {"query": "SELECT * FROM users WHERE id = 'UX341234'"},
            }
        ]
    )


def herramienta(llamada):
    """Imita herramienta_personalizada: usa los args de una llamada."""
    return f"Respuesta a la consulta: {llamada['args']['query']}"


pregunta = "Genera un resumen del usuario UX341234"

## 1. Sin `attrgetter` y sin cadena

Partes el flujo en variables y en un `for`. Es lo que hace el comentario de `herramientas_personalizadas_llm.py`, en varios pasos.

In [ ]:
mensaje = modelo(pregunta)
llamadas = mensaje.tool_calls

resultados = []
for llamada in llamadas:
    resultados.append(herramienta(llamada))

resultados

## 2. En una cadena, escribiendo tú la función del medio

El `|` encadena funciones: la salida de una entra en la siguiente.

Al armar la cadena el mensaje todavía no existe, así que no puedes escribir `mensaje.tool_calls` ahí. Dejas una función que lo lea cuando el mensaje llegue.

`sacar_tool_calls` es esa función. `ejecutar_cada_llamada` es el `for` de antes.

In [ ]:
def sacar_tool_calls(mensaje):
    return mensaje.tool_calls


def ejecutar_cada_llamada(llamadas):
    return [herramienta(llamada) for llamada in llamadas]


resultado = ejecutar_cada_llamada(sacar_tool_calls(modelo(pregunta)))
resultado

## 3. Con `attrgetter`

`attrgetter("tool_calls")` sustituye el `def sacar_tool_calls` de arriba.

En el archivo real, `.map()` sustituye el `for`. La línea queda así:

```python
chain = llm_with_tools | attrgetter("tool_calls") | herramienta_personalizada.map()
```

In [ ]:
sacar_tool_calls = attrgetter("tool_calls")
resultado = ejecutar_cada_llamada(sacar_tool_calls(modelo(pregunta)))
resultado

## Las tres lecturas son la misma lista

`mensaje.tool_calls`, la función escrita a mano y `attrgetter("tool_calls")` devuelven el mismo valor. `attrgetter` no transforma el dato: evita escribir el `def`.

In [ ]:
mensaje = modelo(pregunta)

print("mensaje.tool_calls                 ->", mensaje.tool_calls)
print("sacar_tool_calls(mensaje)           ->", sacar_tool_calls(mensaje))
print('attrgetter("tool_calls")(mensaje)   ->', attrgetter("tool_calls")(mensaje))